# 🗄️ NOVA — GRPO Training Notebook

**Train Qwen2.5-1.5B to be a Senior DBA Agent using GRPO + LoRA**

This notebook trains a small LLM (1.5B params) to optimize SQL query costs through reinforcement learning.
Each episode: reset DB → agent picks index strategy → reward = did query cost drop?

> **Runtime:** GPU (T4 or better). Go to `Runtime → Change runtime type → T4 GPU`
> **Time:** ~20–40 minutes for 60 episodes

## 1. Install Dependencies

In [ ]:
!pip install -q openenv-core>=0.1.13 trl>=0.8.0 peft>=0.10.0 \
    accelerate>=0.28.0 transformers>=4.40.0 datasets>=2.18.0 \
    fastapi uvicorn openai plotly streamlit

## 2. Clone the Repository

In [ ]:
!git clone https://github.com/itsflash44/db-tune-project.git
%cd db-tune-project

## 3. Configuration

In [ ]:
import os

# --- SET THESE ---
ENV_URL      = 'https://itsflash44-db-tune-env.hf.space'  # your HF Space
HF_TOKEN     = ''   # optional: paste your HF token to push trained model
HF_REPO      = ''   # optional: 'yourname/nova-dba-lora'
NUM_EPISODES = 60   # increase for better training

os.environ['ENV_BASE_URL']  = ENV_URL
os.environ['HF_TOKEN']      = HF_TOKEN
os.environ['HF_REPO']       = HF_REPO
os.environ['NUM_EPISODES']  = str(NUM_EPISODES)

print(f'Environment : {ENV_URL}')
print(f'Episodes    : {NUM_EPISODES}')

## 4. Verify Environment Connectivity

In [ ]:
from client import DBEnvClient, DBAction

print(f'Connecting to {ENV_URL}...')
env = DBEnvClient(base_url=ENV_URL)

with env.sync() as sync_env:
    result = sync_env.reset(task='easy')
    obs = result.observation
    print(f'Connected!')
    print(f'Initial cost   : {obs.query_cost}')
    print(f'Storage budget : {obs.storage_budget}')
    print(f'Indices        : {obs.current_indices}')

print('\n✅ Environment ready for training.')

## 5. Import Training Utilities

In [ ]:
import logging
from reward_functions import reward_total, reward_cost_reduction, reward_storage_safety, StepState
from train import make_rollout_func, SYSTEM_PROMPT, OUTPUT_DIR

logging.basicConfig(level=logging.INFO)
print('System prompt (first 200 chars):')
print(SYSTEM_PROMPT[:200])
print('...')
print(f'\nImported: reward_total, make_rollout_func — ready!')

## 6. GRPO Training Setup

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Dataset (each entry = one episode)
dataset = Dataset.from_dict({'prompt': ['Diagnose and optimise this database.'] * NUM_EPISODES})

# GRPO Config
grpo_config = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    logging_steps=5,
    save_steps=20,
    temperature=0.3,
    max_completion_length=256,
    report_to='none',
)

# LoRA Config
lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

# Trainer
trainer = GRPOTrainer(
    model=MODEL_ID,
    processing_class=tokenizer,
    reward_funcs=[lambda prompts, completions, **kw: [1.0] * len(prompts)],
    args=grpo_config,
    train_dataset=dataset,
    rollout_func=make_rollout_func(env, tokenizer, ['easy', 'medium', 'hard']),
    peft_config=lora_config,
)

print('GRPOTrainer initialized ✅')

## 7. Train!

In [ ]:
print('Starting GRPO training...')
print(f'  Model      : {MODEL_ID}')
print(f'  Episodes   : {NUM_EPISODES}')
print(f'  Environment: {ENV_URL}')
print()

try:
    trainer.train()
finally:
    pass  # env stays open until notebook exits

trainer.save_model(str(OUTPUT_DIR))
print(f'\n✅ Model saved to {OUTPUT_DIR}')

## 8. Reward Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

log_path = next(Path('outputs').glob('**/reward_log.csv'), None)
if log_path:
    df = pd.read_csv(log_path)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    for task, grp in df.groupby('task'):
        ax1.plot(grp['episode'], grp['total_reward'], label=task, linewidth=2)
    ax1.axhline(y=1.5, color='green', linestyle='--', label='Target (+1.5)')
    ax1.set_title('Total Reward per Episode')
    ax1.set_xlabel('Episode'); ax1.set_ylabel('Reward'); ax1.legend()

    for task, grp in df.groupby('task'):
        ax2.plot(grp['episode'], grp['final_cost'], label=task, linewidth=2)
    ax2.axhline(y=10, color='green', linestyle='--', label='Target (10.0)')
    ax2.set_title('Final Query Cost per Episode')
    ax2.set_xlabel('Episode'); ax2.set_ylabel('Cost'); ax2.legend()

    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'reward_curve.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Reward curves saved.')
else:
    print('No reward log found — run training first.')

## 9. Push to Hub (Optional)

In [ ]:
# Uncomment to push:
# if HF_REPO:
#     trainer.push_to_hub(repo_id=HF_REPO)
#     print(f'Model pushed to https://huggingface.co/{HF_REPO}')